In [ ]:
"""
HISID Level-1 Model  v3  –  final
====================================
Key theoretical correction:
  BPD social-L2 inference uses BAYESIAN MIXTURE BELIEF UPDATE,
  not gradient descent.  Rationale: bimodal prior implies a
  discrete latent variable (good / bad attachment representation).
  MAP gradient descent cannot faithfully represent a bimodal posterior;
  Bayesian mixture updating does.  This directly implements
  HISID Postulate 3 (splitting as precision-regularisation to a 2-state
  model) as an explicit computational mechanism.

HC social-L2: continuous Gaussian updating (standard gradient descent).
BPD social-L2: mixture weight update + stochastic mode sampling.
Both share: 2-level interoceptive hierarchy (GD), coupling term.
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from scipy.stats import norm, mannwhitneyu, gaussian_kde
from scipy.ndimage import gaussian_filter1d
import warnings
warnings.filterwarnings('ignore')

RNG = np.random.default_rng(2025)

HC_COL  = '#2166AC';  BPD_COL = '#C0392B'
SH_HC   = '#AED6F1';  SH_BPD  = '#F1948A'
THR_COL = '#E67E22';  ACC_COL = '#27AE60'


# ═══════════════════════════════════════════════════════════════════════════════
# PARADIGM
# ═══════════════════════════════════════════════════════════════════════════════
def make_paradigm(n=300, rng=None):
    if rng is None: rng = RNG
    # 3-state Markov: 0=neutral, 1=accept, 2=threat
    T = np.array([[0.75, 0.13, 0.12],
                  [0.22, 0.70, 0.08],
                  [0.28, 0.08, 0.64]])
    val = {0: 0.0, 1: 0.85, 2: -1.0}
    state = 0; ev = np.zeros(n,int); ss = np.zeros(n)
    for t in range(n):
        ev[t]=state; ss[t]=val[state]
        state=rng.choice(3,p=T[state])
    si = np.zeros(n)
    for t in range(1,n):
        mu_e = 0.35*float(ss[t-1]<-0.5)
        si[t] = si[t-1]+0.15*(mu_e-si[t-1])+0.22*rng.standard_normal()
    oi = si + 0.28*rng.standard_normal(n)
    os = ss + 0.20*rng.standard_normal(n)
    return si, ss, oi, os, ev


# ═══════════════════════════════════════════════════════════════════════════════
# HC AGENT  –  hierarchical Gaussian GD
# ═══════════════════════════════════════════════════════════════════════════════
class HCAgent:
    def __init__(self, pi_int=2.0, pi_int2=1.5,
                 pi_soc=1.5, pi_soc2=1.0, lam=0.30,
                 lr=0.06, n_steps=40):
        self.pi_int=pi_int; self.pi_int2=pi_int2
        self.pi_soc=pi_soc; self.pi_soc2=pi_soc2
        self.lam=lam; self.lr=lr; self.n_steps=n_steps
        self.reset()

    def reset(self):
        self.mu_int1=self.mu_int2=self.mu_soc1=self.mu_soc2=0.0

    def _F(self, oi, os):
        Fi = (0.5*self.pi_int*(oi-self.mu_int1)**2
             +0.5*self.pi_int2*(self.mu_int1-self.mu_int2)**2)
        Fs = (0.5*self.pi_soc*(os-self.mu_soc1)**2
             +0.5*self.pi_soc2*(self.mu_soc1-self.mu_soc2)**2
             +0.5*self.pi_soc2*self.mu_soc2**2)   # Gaussian prior N(0,1/pi_soc2)
        Fc = 0.5*self.lam*(self.mu_int2-self.mu_soc2)**2
        return Fi+Fs+Fc

    def step(self, oi, os, rng=None):
        for _ in range(self.n_steps):
            e1i=oi-self.mu_int1; e2i=self.mu_int1-self.mu_int2
            e1s=os-self.mu_soc1; e2s=self.mu_soc1-self.mu_soc2
            g1i = -self.pi_int*e1i  + self.pi_int2*e2i
            g2i = -self.pi_int2*e2i + self.lam*(self.mu_int2-self.mu_soc2)
            g1s = -self.pi_soc*e1s  + self.pi_soc2*e2s
            g2s = (-self.pi_soc2*e2s + self.pi_soc2*self.mu_soc2
                   - self.lam*(self.mu_int2-self.mu_soc2))
            self.mu_int1 -= self.lr*g1i
            self.mu_int2 -= self.lr*g2i
            self.mu_soc1 -= self.lr*g1s
            self.mu_soc2 -= self.lr*g2s
        return self._F(oi, os)

    @property
    def mu_self(self):
        w_i=self.pi_int2/(self.pi_int2+self.pi_soc)
        return w_i*self.mu_int2 + (1-w_i)*self.mu_soc2


# ═══════════════════════════════════════════════════════════════════════════════
# BPD AGENT  –  suppressed interoception + Bayesian bimodal social inference
# ═══════════════════════════════════════════════════════════════════════════════
class BPDAgent:
    """
    Implements HISID Postulates 1–4:
      1. Precision dissociation:
           pi_int  = 0.35  (suppressed)
           pi_soc_threat elevated on negative observations
      2. Self-model anchoring failure:
           w_soc >> w_int because pi_int2 is tiny
      3. Splitting via bimodal mixture belief:
           tracks P(partner=good) continuously; samples categorical state
           → produces discrete ±v switches with high variance
      4. Prior learned from adversity (initialised w_plus = 0.4,
           slight bias toward threat-expectancy)
    """
    def __init__(self,
                 # interoceptive (suppressed)
                 pi_int=0.35,   pi_int2=0.22,
                 # social
                 pi_soc_base=1.5,  pi_soc_threat=3.0,
                 obs_thresh=-0.20,
                 # social L2 bimodal prior
                 v_plus=0.80, v_minus=-0.80,
                 # coupling + GD for interoceptive hierarchy
                 lam=0.30, lr=0.06, n_steps=40):

        self.pi_int=pi_int; self.pi_int2=pi_int2
        self.pi_soc_base=pi_soc_base; self.pi_soc_threat=pi_soc_threat
        self.obs_thresh=obs_thresh
        self.v_plus=v_plus; self.v_minus=v_minus
        self.lam=lam; self.lr=lr; self.n_steps=n_steps
        self.reset()

    def reset(self):
        self.mu_int1=self.mu_int2=0.0
        self.mu_soc1=0.0
        # mixture weight on positive pole (adversity-driven prior: slight -ve bias)
        self.w_plus = 0.40
        # current realised social state (sampled from mixture)
        self.mu_soc2 = self.v_minus  # start with threat-expectancy

    def _pi_eff(self, os):
        return self.pi_soc_base + (self.pi_soc_threat if os < self.obs_thresh else 0.0)

    # ── interoceptive hierarchy (standard GD) ─────────────────────────────
    def _step_interoceptive(self, oi):
        for _ in range(self.n_steps):
            e1 = oi - self.mu_int1
            e2 = self.mu_int1 - self.mu_int2
            g1 = -self.pi_int*e1  + self.pi_int2*e2
            g2 = -self.pi_int2*e2 + self.lam*(self.mu_int2-self.mu_soc2)
            self.mu_int1 -= self.lr*g1
            self.mu_int2 -= self.lr*g2

    # ── social L2: Bayesian mixture belief update (splitting) ─────────────
    def _step_social_mixture(self, os, rng):
        pi = self._pi_eff(os)
        # log-likelihood of observation under each pole (marginal over L1)
        # Approximate marginal: p(o_soc | mu_soc2=v) ~ N(o_soc; v, 1/pi_soc_base)
        # Threat-detection uses boosted precision pi
        log_lik_plus  = -0.5*pi*(os - self.v_plus)**2
        log_lik_minus = -0.5*pi*(os - self.v_minus)**2

        # Bayesian update of mixture weights
        log_post_plus  = np.log(self.w_plus)       + log_lik_plus
        log_post_minus = np.log(1.0 - self.w_plus) + log_lik_minus
        lZ = np.logaddexp(log_post_plus, log_post_minus)
        self.w_plus = float(np.clip(np.exp(log_post_plus - lZ), 1e-6, 1-1e-6))

        # Sample current mode from posterior mixture → splitting
        self.mu_soc2 = (self.v_plus
                        if rng.random() < self.w_plus
                        else self.v_minus)

        # L1 social state: tracks actual observation (sensory layer)
        for _ in range(self.n_steps):
            e1 = os - self.mu_soc1
            g1 = -self._pi_eff(os)*e1 + self.pi_int2*(self.mu_soc1-self.mu_soc2)
            self.mu_soc1 -= self.lr*g1

    def free_energy(self, oi, os):
        pi=self._pi_eff(os)
        Fi=(0.5*self.pi_int*(oi-self.mu_int1)**2
           +0.5*self.pi_int2*(self.mu_int1-self.mu_int2)**2)
        Fs=(0.5*pi*(os-self.mu_soc1)**2
           +0.5*self.pi_int2*(self.mu_soc1-self.mu_soc2)**2)
        # mixture prior surprise (entropy term)
        Fprior=-( self.w_plus*np.log(self.w_plus+1e-9)
                +(1-self.w_plus)*np.log(1-self.w_plus+1e-9))
        Fc=0.5*self.lam*(self.mu_int2-self.mu_soc2)**2
        return Fi+Fs+Fprior+Fc

    def step(self, oi, os, rng=None):
        if rng is None: rng=np.random.default_rng()
        self._step_interoceptive(oi)
        self._step_social_mixture(os, rng)
        return self.free_energy(oi, os)

    @property
    def mu_self(self):
        # HISID Postulate 2: pi_int2 tiny → w_soc dominates
        w_i=self.pi_int2/(self.pi_int2+self.pi_soc_base)
        return w_i*self.mu_int2 + (1-w_i)*self.mu_soc2


# ═══════════════════════════════════════════════════════════════════════════════
# SIMULATION UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════
def run_sim(agent, oi, os, ev, rng=None):
    n=len(oi)
    h={k:np.zeros(n) for k in
       ['mu_self','mu_int2','mu_soc2','mu_int1','mu_soc1','free_energy','w_plus']}
    agent.reset()
    for t in range(n):
        h['free_energy'][t]=agent.step(oi[t],os[t],rng)
        h['mu_self'][t]=agent.mu_self
        h['mu_int2'][t]=agent.mu_int2
        h['mu_soc2'][t]=agent.mu_soc2
        h['mu_soc1'][t]=agent.mu_soc1
        h['mu_int1'][t]=agent.mu_int1
        h['w_plus'][t]=getattr(agent,'w_plus',0.5)
    return h

def recovery_times(mu_soc2, ev, threshold=-0.05):
    threat_on=np.where(np.diff((ev==2).astype(int))==1)[0]+1
    rts=[]
    for t0 in threat_on:
        t=t0
        while t<len(mu_soc2) and mu_soc2[t]<threshold: t+=1
        rt=t-t0
        if 0<rt<len(mu_soc2)-t0: rts.append(rt)
    return rts

def ensemble(AgentClass, N=40, n=300, **kwargs):
    mus,msoc,fe,rts,wplus=[],[],[],[],[]
    for i in range(N):
        rng_i=np.random.default_rng(5000+i)
        _,_,oi,os,ev=make_paradigm(n,rng_i)
        ag=AgentClass(**kwargs)
        h=run_sim(ag,oi,os,ev,rng_i)
        mus.append(h['mu_self'])
        msoc.append(h['mu_soc2'])
        fe.append(h['free_energy'])
        rts.extend(recovery_times(h['mu_soc2'],ev))
        wplus.append(h['w_plus'])
    return (np.array(mus),np.array(msoc),
            np.array(fe),rts,
            np.array(wplus) if wplus[0].ndim==1 else None)


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 – Main results
# ═══════════════════════════════════════════════════════════════════════════════
def figure_main():
    print("HC  ensemble (n=40)...")
    ms_hc, msoc_hc, fe_hc, rt_hc, _ = ensemble(HCAgent,  40)
    print("BPD ensemble (n=40)...")
    ms_bpd,msoc_bpd,fe_bpd,rt_bpd,wp_bpd = ensemble(BPDAgent, 40)

    # representative run
    _,ss,oi,os,ev=make_paradigm(300, np.random.default_rng(77))
    rng77=np.random.default_rng(77)
    hc_r =run_sim(HCAgent(),  oi,os,ev)
    bpd_r=run_sim(BPDAgent(),oi,os,ev,np.random.default_rng(77))
    T=np.arange(300)
    tm=(ev==2); am=(ev==1)

    fe_hc_cum =np.cumsum(fe_hc, axis=1)
    fe_bpd_cum=np.cumsum(fe_bpd,axis=1)

    sm=lambda x,s=2: gaussian_filter1d(x,s)

    fig=plt.figure(figsize=(17,21))
    fig.patch.set_facecolor('white')
    gs=gridspec.GridSpec(4,3,figure=fig,hspace=0.55,wspace=0.42,
                         left=0.07,right=0.97,top=0.94,bottom=0.05)

    # ── A: self-model trajectory ───────────────────────────────────────────
    ax=fig.add_subplot(gs[0,:2])
    for t in range(299):
        if tm[t]: ax.axvspan(t,t+1,color=THR_COL,alpha=0.12,lw=0)
        if am[t]: ax.axvspan(t,t+1,color=ACC_COL,alpha=0.08,lw=0)
    ax.plot(T,sm(hc_r['mu_self']), color=HC_COL, lw=2,label='HC',zorder=3)
    ax.plot(T,sm(bpd_r['mu_self'],1),color=BPD_COL,lw=2,label='BPD',zorder=3,alpha=0.88)
    ax.axhline(0,color='grey',lw=0.7,ls='--')
    ax.axhline( 0.80*0.78, color=BPD_COL,lw=0.8,ls=':',alpha=0.5)  # idealisation level
    ax.axhline(-0.80*0.78, color=BPD_COL,lw=0.8,ls=':',alpha=0.5)  # devaluation level
    ax.set_xlim(0,299); ax.set_xlabel('Trial',fontsize=11)
    ax.set_ylabel(r'$\mu_{\mathrm{self}}$',fontsize=12)
    ax.set_title('A — Self-model trajectory  (representative run)',
                 fontsize=11,fontweight='bold',loc='left')
    ax.legend(handles=[
        Line2D([0],[0],color=HC_COL, lw=2,label='HC (smooth tracking)'),
        Line2D([0],[0],color=BPD_COL,lw=2,label='BPD (splitting oscillation)'),
        Patch(color=THR_COL,alpha=0.3,label='Threat period'),
        Patch(color=ACC_COL,alpha=0.3,label='Accept period'),
    ],fontsize=9,framealpha=0.9,loc='lower right',ncol=2)

    # ── B: self-model variance ─────────────────────────────────────────────
    ax=fig.add_subplot(gs[0,2])
    vhc =np.var(ms_hc, axis=1)
    vbpd=np.var(ms_bpd,axis=1)
    vp=ax.violinplot([vhc,vbpd],positions=[1,2],showmedians=True,widths=0.55)
    for i,body in enumerate(vp['bodies']):
        body.set_facecolor([HC_COL,BPD_COL][i]); body.set_alpha(0.70)
    for p in ['cbars','cmins','cmaxes','cmedians']:
        vp[p].set_color('k'); vp[p].set_linewidth(1.3)
    ax.set_xticks([1,2]); ax.set_xticklabels(['HC','BPD'],fontsize=11)
    ax.set_ylabel(r'$\mathrm{Var}(\mu_{\mathrm{self}})$',fontsize=11)
    ax.set_title('B — Self-model variance\n(n=40 per group)',
                 fontsize=11,fontweight='bold',loc='left')
    ratio_v=np.median(vbpd)/np.median(vhc)
    ax.text(1.5,np.percentile(vbpd,80),f'{ratio_v:.1f}×',
            ha='center',fontsize=13,fontweight='bold',color=BPD_COL)

    # ── C: partner-valence distribution ───────────────────────────────────
    ax=fig.add_subplot(gs[1,:2])
    v=np.linspace(-1.4,1.4,500)
    kdehc =gaussian_kde(hc_r['mu_soc2'], bw_method=0.25)
    # BPD: use all ensemble data for a cleaner distribution
    all_bpd_soc=msoc_bpd.flatten()
    kdebpd=gaussian_kde(all_bpd_soc, bw_method=0.15)
    ax.fill_between(v,kdehc(v), color=HC_COL, alpha=0.40)
    ax.fill_between(v,kdebpd(v),color=BPD_COL,alpha=0.40)
    ax.plot(v,kdehc(v), color=HC_COL, lw=2.2,label='HC  (unimodal)')
    ax.plot(v,kdebpd(v),color=BPD_COL,lw=2.2,label='BPD (bimodal = splitting)')
    ax.axvline( 0.80,color='grey',ls=':',lw=1.5,label='Prior poles (±0.80)')
    ax.axvline(-0.80,color='grey',ls=':',lw=1.5)
    ax.set_xlabel(r'Partner-valence belief $\mu_{\mathrm{soc}}^{(2)}$',fontsize=11)
    ax.set_ylabel('Density',fontsize=11)
    ax.set_title('C — Partner-valence posterior  (Postulate 3: bimodal = splitting)',
                 fontsize=11,fontweight='bold',loc='left')
    ax.legend(fontsize=10,framealpha=0.9)

    # ── D: recovery time ──────────────────────────────────────────────────
    ax=fig.add_subplot(gs[1,2])
    mx=min(max(rt_hc+rt_bpd,default=1),55)
    bins=np.linspace(0,mx,15)
    if rt_hc:
        ax.hist(rt_hc, bins=bins,color=HC_COL, alpha=0.70,
                edgecolor='w',lw=0.5,density=True,
                label=f'HC   μ={np.mean(rt_hc):.1f} trials')
    if rt_bpd:
        ax.hist(rt_bpd,bins=bins,color=BPD_COL,alpha=0.70,
                edgecolor='w',lw=0.5,density=True,
                label=f'BPD  μ={np.mean(rt_bpd):.1f} trials')
    if rt_hc:  ax.axvline(np.mean(rt_hc), color=HC_COL, ls='--',lw=2)
    if rt_bpd: ax.axvline(np.mean(rt_bpd),color=BPD_COL,ls='--',lw=2)
    ax.set_xlabel('Recovery time (trials)',fontsize=11)
    ax.set_ylabel('Density',fontsize=11)
    ax.set_title('D — Threat recovery time\n(Postulate 1: threat amplification)',
                 fontsize=11,fontweight='bold',loc='left')
    ax.legend(fontsize=9,framealpha=0.9)
    if rt_hc and rt_bpd:
        ax.text(0.97,0.97,f'ratio: {np.mean(rt_bpd)/np.mean(rt_hc):.1f}×',
                transform=ax.transAxes,ha='right',va='top',fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3',fc='lightyellow',ec='grey',lw=0.7))

    # ── E: cumulative free energy ──────────────────────────────────────────
    ax=fig.add_subplot(gs[2,:2])
    mhc =np.mean(fe_hc_cum, axis=0); shc =np.std(fe_hc_cum, axis=0)
    mbpd=np.mean(fe_bpd_cum,axis=0); sbpd=np.std(fe_bpd_cum,axis=0)
    ax.fill_between(T,mhc-shc,  mhc+shc,  color=SH_HC, alpha=0.55)
    ax.fill_between(T,mbpd-sbpd,mbpd+sbpd,color=SH_BPD,alpha=0.55)
    ax.plot(T,mhc, color=HC_COL, lw=2,label='HC  (mean±SD)')
    ax.plot(T,mbpd,color=BPD_COL,lw=2,label='BPD (mean±SD)')
    ax.set_xlabel('Trial',fontsize=11)
    ax.set_ylabel(r'Cumulative $\mathcal{F}$',fontsize=12)
    ax.set_title('E — Cumulative free energy  (allostatic load proxy, n=40)',
                 fontsize=11,fontweight='bold',loc='left')
    ax.legend(fontsize=10,framealpha=0.9)
    ax.set_xlim(0,299)
    ratio_fe=np.mean(fe_bpd_cum[:,-1])/np.mean(fe_hc_cum[:,-1])
    ax.text(0.02,0.97,f'BPD/HC cumulative F ratio = {ratio_fe:.2f}×',
            transform=ax.transAxes,va='top',fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3',fc='lightyellow',ec='grey',lw=0.7))

    # ── F: phase portrait ─────────────────────────────────────────────────
    ax=fig.add_subplot(gs[2,2])
    ax.scatter(hc_r['mu_int2'], hc_r['mu_soc2'],
               c=T,cmap='Blues',s=10,alpha=0.70,zorder=2)
    ax.scatter(bpd_r['mu_int2'],bpd_r['mu_soc2'],
               c=T,cmap='Reds', s=10,alpha=0.70,zorder=2)
    ax.axhline(0,color='grey',lw=0.6,ls='--')
    ax.axvline(0,color='grey',lw=0.6,ls='--')
    ax.axhline( 0.80,color=BPD_COL,lw=0.8,ls=':',alpha=0.5)
    ax.axhline(-0.80,color=BPD_COL,lw=0.8,ls=':',alpha=0.5)
    ax.set_xlabel(r'$\mu_{\mathrm{int}}^{(2)}$',fontsize=11)
    ax.set_ylabel(r'$\mu_{\mathrm{soc}}^{(2)}$',fontsize=11)
    ax.set_title('F — Phase portrait\n(int. × soc. beliefs)',
                 fontsize=11,fontweight='bold',loc='left')
    ax.legend(handles=[
        Line2D([0],[0],marker='o',color='w',markerfacecolor=HC_COL, ms=7,label='HC'),
        Line2D([0],[0],marker='o',color='w',markerfacecolor=BPD_COL,ms=7,label='BPD'),
    ],fontsize=10,framealpha=0.9)

    # ── G: trial-by-trial F ───────────────────────────────────────────────
    ax=fig.add_subplot(gs[3,:2])
    for t in range(299):
        if tm[t]: ax.axvspan(t,t+1,color=THR_COL,alpha=0.10,lw=0)
    mhct =sm(np.mean(fe_hc, axis=0),6); sehct =sm(np.std(fe_hc, axis=0)/np.sqrt(40),6)
    mbpdt=sm(np.mean(fe_bpd,axis=0),6); sebpdt=sm(np.std(fe_bpd,axis=0)/np.sqrt(40),6)
    ax.fill_between(T,mhct-sehct,  mhct+sehct,  color=SH_HC, alpha=0.6)
    ax.fill_between(T,mbpdt-sebpdt,mbpdt+sebpdt,color=SH_BPD,alpha=0.6)
    ax.plot(T,mhct, color=HC_COL, lw=2,label='HC  (mean±SE)')
    ax.plot(T,mbpdt,color=BPD_COL,lw=2,label='BPD (mean±SE)')
    ax.set_xlabel('Trial',fontsize=11)
    ax.set_ylabel(r'$\mathcal{F}$ per trial',fontsize=12)
    ax.set_title('G — Per-trial free energy  (threat periods shaded)',
                 fontsize=11,fontweight='bold',loc='left')
    ax.legend(fontsize=10,framealpha=0.9); ax.set_xlim(0,299)

    # ── H: mixture weight (w+) over time ──────────────────────────────────
    ax=fig.add_subplot(gs[3,2])
    # compute w_plus trajectory on representative BPD run
    bpd_wp=bpd_r['w_plus']
    ax.plot(T,sm(bpd_wp,3),color=BPD_COL,lw=2,label='BPD: P(partner=good)')
    ax.plot(T,sm(0.5+0.0*T,1),color=HC_COL,lw=2,ls='--',
            label='HC: implicit $P \\approx 0.5$ (no poles)')
    for t in range(299):
        if tm[t]: ax.axvspan(t,t+1,color=THR_COL,alpha=0.10,lw=0)
    ax.axhline(0.5,color='grey',lw=0.7,ls=':')
    ax.set_ylim(-0.05,1.05)
    ax.set_xlabel('Trial',fontsize=11)
    ax.set_ylabel(r'$w_+ = P(\mathrm{partner\ good})$',fontsize=10)
    ax.set_title('H — Splitting dynamics:\nmixture weight $w_+$',
                 fontsize=11,fontweight='bold',loc='left')
    ax.legend(fontsize=9,framealpha=0.9); ax.set_xlim(0,299)

    fig.suptitle(
        'HISID Level-1 Simulation Results\n'
        'BPD agent (precision-dissociated, bimodal splitting) vs. HC agent  '
        '—  300-trial social-threat paradigm  (n = 40)',
        fontsize=13,fontweight='bold',y=0.97)
    plt.savefig('/home/claude/hisid_results.pdf',dpi=300,
                bbox_inches='tight',facecolor='white')
    plt.savefig('/home/claude/hisid_results.png',dpi=180,
                bbox_inches='tight',facecolor='white')
    print("Main figure saved.")

    # ── stats ──────────────────────────────────────────────────────────────
    vhc2=np.var(ms_hc,axis=1); vbpd2=np.var(ms_bpd,axis=1)
    print("\n" + "="*64)
    print("SUMMARY STATISTICS")
    print("="*64)
    print(f"Self-model variance  HC={np.median(vhc2):.4f}  BPD={np.median(vbpd2):.4f}  "
          f"ratio={np.median(vbpd2)/np.median(vhc2):.2f}x")
    if rt_hc and rt_bpd:
        print(f"Recovery time        HC={np.mean(rt_hc):.2f}  BPD={np.mean(rt_bpd):.2f}  "
              f"ratio={np.mean(rt_bpd)/np.mean(rt_hc):.2f}x")
    elif rt_bpd:
        print(f"Recovery time        BPD={np.mean(rt_bpd):.2f}  (HC: immediate)")
    else:
        print("Recovery time        BPD: no recovery observed")
    print(f"Cumulative F at T300 HC={np.mean(fe_hc_cum[:,-1]):.2f}  "
          f"BPD={np.mean(fe_bpd_cum[:,-1]):.2f}  "
          f"ratio={np.mean(fe_bpd_cum[:,-1])/np.mean(fe_hc_cum[:,-1]):.2f}x")
    u1,p1=mannwhitneyu(vhc2,vbpd2,alternative='less')
    u3,p3=mannwhitneyu(fe_hc_cum[:,-1],fe_bpd_cum[:,-1],alternative='less')
    print(f"Mann-Whitney (HC<BPD): variance U={u1:.0f} p={p1:.3e}  "
          f"cumF U={u3:.0f} p={p3:.3e}")
    if rt_hc and rt_bpd:
        u2,p2=mannwhitneyu(rt_hc,rt_bpd,alternative='less')
        print(f"                       recov.  U={u2:.0f} p={p2:.3e}")
    print("="*64)
    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 – Free-energy landscape
# ═══════════════════════════════════════════════════════════════════════════════
def figure_landscape():
    fig=plt.figure(figsize=(14,6)); fig.patch.set_facecolor('white')
    oi_f, os_f = 0.2, -0.65   # mild threat

    mi=np.linspace(-1.4,1.4,65); ms=np.linspace(-1.8,1.8,65)
    MI,MS=np.meshgrid(mi,ms)

    configs=[
        ('Healthy Control  (Gaussian prior)',  HCAgent,  {},              'Blues'),
        ('BPD  (Bimodal / splitting prior)',   BPDAgent, {'w_plus':0.5},  'Reds'),
    ]
    for idx,(label,Cls,extra,cmap) in enumerate(configs):
        ag=Cls(**{k:v for k,v in extra.items() if k not in ['w_plus']})
        # for BPD, set w_plus manually
        if hasattr(ag,'w_plus'): ag.w_plus=extra.get('w_plus',0.5)
        Z=np.zeros_like(MI)
        for i in range(MI.shape[0]):
            for j in range(MI.shape[1]):
                ag.mu_int2=ag.mu_int1=float(MI[i,j])
                ag.mu_soc2=float(MS[i,j])
                ag.mu_soc1=float(MS[i,j])
                if hasattr(ag,'w_plus'):
                    # compute mixture-based F analytically at fixed mu_soc2
                    pi=ag._pi_eff(os_f)
                    Fi=(0.5*ag.pi_int*(oi_f-ag.mu_int1)**2
                       +0.5*ag.pi_int2*(ag.mu_int1-ag.mu_int2)**2)
                    # Social surprise: mixture prior = -log[w+*N(μ;v+)+w-*N(μ;v-)]
                    lp=(np.log(ag.w_plus)
                        +norm.logpdf(float(MS[i,j]),ag.v_plus,0.30))
                    lm=(np.log(1-ag.w_plus)
                        +norm.logpdf(float(MS[i,j]),ag.v_minus,0.30))
                    Fprior=-np.logaddexp(lp,lm)
                    Fs=(0.5*pi*(os_f-ag.mu_soc1)**2+Fprior)
                    Fc=0.5*ag.lam*(ag.mu_int2-ag.mu_soc2)**2
                    Z[i,j]=Fi+Fs+Fc
                else:
                    Z[i,j]=ag._F(oi_f,os_f)
        Z=np.clip(Z,0,np.percentile(Z,93))
        ax=fig.add_subplot(1,2,idx+1,projection='3d')
        surf=ax.plot_surface(MI,MS,Z,cmap=cmap,alpha=0.88,
                             linewidth=0,antialiased=True,rstride=2,cstride=2)
        ax.set_xlabel(r'$\mu_{\mathrm{int}}^{(2)}$',fontsize=10,labelpad=5)
        ax.set_ylabel(r'$\mu_{\mathrm{soc}}^{(2)}$',fontsize=10,labelpad=5)
        ax.set_zlabel(r'$\mathcal{F}$',fontsize=11,labelpad=5)
        ax.set_title(label,fontsize=11,fontweight='bold',pad=8)
        ax.view_init(elev=30,azim=-52)
        fig.colorbar(surf,ax=ax,shrink=0.45,pad=0.06,label=r'$\mathcal{F}$')
    fig.suptitle(
        r'Free-Energy Landscape  $\mathcal{F}(\mu_{\mathrm{int}}^{(2)},\,\mu_{\mathrm{soc}}^{(2)})$'
        '\n$o_{\\mathrm{int}}=0.2$,  $o_{\\mathrm{soc}}=-0.65$  (mild threat)'
        '\nBPD: double-well in social dimension  →  splitting attractor  (HISID Postulate 3)',
        fontsize=12,fontweight='bold')
    plt.tight_layout()
    plt.savefig('/home/claude/hisid_landscape.pdf',dpi=300,
                bbox_inches='tight',facecolor='white')
    plt.savefig('/home/claude/hisid_landscape.png',dpi=180,
                bbox_inches='tight',facecolor='white')
    print("Landscape figure saved.")
    return fig


if __name__=='__main__':
    print("HISID v3 — Level-1 Simulation")
    print("="*64)
    f1=figure_main()
    f2=figure_landscape()
    print("Complete.")
